In [1]:
from pymargo.core import Engine
import pyyokan_common as yokan
from pyyokan_client import Client
from pyyokan_server import Provider

In [2]:
import json
import ctypes
import struct
import blosc2
import numpy as np

In [3]:
f = open('../run/mochi-yokan-config.json')
json_data = json.load(f)
json_data

{'sim-id': '00001',
 'libraries': {'yokan': '/vast/home/pascalgrosset/spack/opt/spack/linux-rhel8-haswell/gcc-9.4.0/mochi-yokan-0.4.2-hiu7yh7om6nmyc2ahuknpdsov5k64zcj/lib/libyokan-bedrock-module.so'},
 'providers': [{'name': 'yokan_provider',
   'provider_id': 123,
   'type': 'yokan',
   'pool': '__primary__',
   'config': {'database': {'type': 'map'}}}],
 'databases': [{'address': '192.168.81.98:46089',
   'protocol': 'ofi+tcp',
   'provider_id': 123},
  {'address': '192.168.81.100:33977',
   'protocol': 'ofi+tcp',
   'provider_id': 123}]}

In [ ]:
json_data['providers'][0]['config']

In [ ]:
dbs = []
for db in json_data['databases']:
    address = db['address']
    protocol = db['protocol']
    provider_id = db['provider_id']
    server_addr = protocol + '://' + address
    print(server_addr)
    
    engine = Engine(protocol)
    mid = engine.get_internal_mid()
    addr = engine.lookup(server_addr)
    hg_addr = addr.get_internal_hg_addr()
    provider = Provider(mid=mid, provider_id=provider_id, config='{"database":{"type":"map"}}')
    client = Client(mid=mid)
    db = client.make_database_handle(address=hg_addr, provider_id=provider_id)
    
    dbs.append(db)
    
   

In [4]:
server_addr1 = "ofi+tcp://192.168.81.98:46089"
server_addr2 = "ofi+tcp://192.168.81.100:33977"
provider_id = 123
protocol = 'ofi+tcp'

In [5]:
engine1 = Engine(protocol)
mid1 = engine1.get_internal_mid()
addr1 = engine1.lookup(server_addr1)
hg_addr1 = addr1.get_internal_hg_addr()
provider1 = Provider(mid=mid1, provider_id=provider_id, config='{"database":{"type":"map"}}')
client1 = Client(mid=mid1)
db1 = client1.make_database_handle(address=hg_addr1, provider_id=provider_id)

In [7]:
engine2 = Engine(protocol)
mid2 = engine2.get_internal_mid()
addr2 = engine2.lookup(server_addr2)
hg_addr2 = addr2.get_internal_hg_addr()
provider2 = Provider(mid=mid2, provider_id=provider_id, config='{"database":{"type":"map"}}')
client2 = Client(mid=mid2)
db2 = client2.make_database_handle(address=hg_addr2, provider_id=provider_id)

In [9]:
dbs = []

In [10]:
dbs.append(db1)
dbs.append(db2)

In [ ]:
# Params
provider_id = 123
#protocol = 'na+sm'
protocol = 'ofi+tcp'
#server_addr = 'na+sm://1364645-0'
server_addr = 'ofi+tcp://192.168.81.80:38989'

In [ ]:
engine = Engine(protocol)
mid = engine.get_internal_mid()
addr = engine.lookup(server_addr)
hg_addr = addr.get_internal_hg_addr()
provider = Provider(mid=mid, provider_id=provider_id, config='{"database":{"type":"map"}}')
client = Client(mid=mid)
db = client.make_database_handle(address=hg_addr, provider_id=provider_id)

In [11]:
ts = '_2'

In [12]:
def list_all_keys(db):
    num_keys = db.count()
    
    max_length = 1024
    prefix = ''
    out_keys = []
    for i in range(0, num_keys):
      out_keys.append( bytearray(max_length+len(prefix)+1) )
    
    from_key = ''
    ksizes = db.list_keys(keys=out_keys, from_key=from_key, filter=prefix)
    
    keys = []
    for i in range(len(ksizes)):
        key_size = ksizes[i]
        
        k = out_keys[i]
        key = (k[:key_size]).decode('ascii')
        keys.append(key)
        
    return keys

In [13]:
def split_key(key, pos):
    parts = key.split('/')
    name = parts[pos]
    return name

In [14]:
def get_field_name(key):
    parts = keys[0].split('/')
    name = parts[len(parts)-2]
    return name

In [15]:
def list_fields(db, timestep):
    
    all_keys = list_all_keys(db)
    print(all_keys)
    
    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[len(parts)-2]
        x.append(name)
    return list(set(x))

In [16]:
def list_attributes(db, key):
    all_keys = list_all_keys(db)

    x = []
    for k in all_keys:
        parts = k.split('/')
        name = parts[2]
        field = parts[3]
        if name == key:
            x.append(field)
    x = list(set(x))

    return x

In [17]:
def get_value(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    v = out_val.decode("ascii")     # convert to ascii
    return v

In [18]:
def get_data(db, key):
    ''' Get data from the server for that key '''

    # length of the value associated with the key
    l = db.length(key)

    out_val = bytearray(l)          # create buffer
    db.get(key=key, value=out_val)  # get the data
    return out_val

In [19]:
keys = list_fields(dbs[0], ts)
keys

['_0/0/pressure_3/compressed_size', '_0/0/pressure_3/num_elems', '_0/0/pressure_3/type', '_0/0/pressure_3/value', '_0/0/temperature_3/compressed_size', '_0/0/temperature_3/num_elems', '_0/0/temperature_3/type', '_0/0/temperature_3/value', '_0/status', '_10/0/pressure_3/compressed_size', '_10/0/pressure_3/num_elems', '_10/0/pressure_3/type', '_10/0/pressure_3/value', '_10/0/temperature_3/compressed_size', '_10/0/temperature_3/num_elems', '_10/0/temperature_3/type', '_10/0/temperature_3/value', '_10/status', '_12/0/pressure_3/compressed_size', '_12/0/pressure_3/num_elems', '_12/0/pressure_3/type', '_12/0/pressure_3/value', '_12/0/temperature_3/compressed_size', '_12/0/temperature_3/num_elems', '_12/0/temperature_3/type', '_12/0/temperature_3/value', '_12/status', '_14/0/pressure_3/compressed_size', '_14/0/pressure_3/num_elems', '_14/0/pressure_3/type', '_14/0/pressure_3/value', '_14/0/temperature_3/compressed_size', '_14/0/temperature_3/num_elems', '_14/0/temperature_3/type', '_14/0/temp

['_34',
 '_54',
 '_18',
 '_8',
 '_64',
 '_14',
 '_16',
 '_56',
 '_42',
 '_68',
 '_26',
 '_4',
 'pressure_3',
 '_60',
 '_20',
 '_28',
 '_32',
 '_30',
 '_22',
 '_10',
 '_52',
 '_46',
 '_2',
 '_12',
 '_62',
 'temperature_3',
 '_40',
 '_44',
 '_38',
 '_66',
 '_50',
 '_58',
 '_24',
 '_48',
 '_6',
 '_36',
 '_0']

In [20]:
keys2 = list_fields(dbs[1], ts)
keys2

['_1/0/pressure_3/compressed_size', '_1/0/pressure_3/num_elems', '_1/0/pressure_3/type', '_1/0/pressure_3/value', '_1/0/temperature_3/compressed_size', '_1/0/temperature_3/num_elems', '_1/0/temperature_3/type', '_1/0/temperature_3/value', '_1/status', '_11/0/pressure_3/compressed_size', '_11/0/pressure_3/num_elems', '_11/0/pressure_3/type', '_11/0/pressure_3/value', '_11/0/temperature_3/compressed_size', '_11/0/temperature_3/num_elems', '_11/0/temperature_3/type', '_11/0/temperature_3/value', '_11/status', '_13/0/pressure_3/compressed_size', '_13/0/pressure_3/num_elems', '_13/0/pressure_3/type', '_13/0/pressure_3/value', '_13/0/temperature_3/compressed_size', '_13/0/temperature_3/num_elems', '_13/0/temperature_3/type', '_13/0/temperature_3/value', '_13/status', '_15/0/pressure_3/compressed_size', '_15/0/pressure_3/num_elems', '_15/0/pressure_3/type', '_15/0/pressure_3/value', '_15/0/temperature_3/compressed_size', '_15/0/temperature_3/num_elems', '_15/0/temperature_3/type', '_15/0/temp

['_57',
 '_5',
 '_43',
 '_15',
 '_19',
 '_31',
 '_25',
 '_59',
 '_41',
 'pressure_3',
 '_13',
 '_63',
 '_67',
 '_35',
 '_29',
 '_21',
 '_39',
 '_53',
 '_55',
 '_23',
 '_45',
 '_7',
 '_9',
 '_3',
 '_61',
 '_69',
 '_37',
 '_65',
 'temperature_3',
 '_51',
 '_27',
 '_11',
 '_47',
 '_1',
 '_33',
 '_17',
 '_49']

In [ ]:
attributes = list_attributes(db, 'pressure_2')
attributes

In [27]:
from mpi4py import MPI
import sys

In [28]:
comm = MPI.COMM_SELF.Spawn(sys.executable, maxprocs=5)
size = comm.Get_size()
rank = comm.Get_rank()

--------------------------------------------------------------------------
A request has timed out and will therefore fail:

  Operation:  LOOKUP: orted/pmix/pmix_server_pub.c:345

Your job may terminate as a result of this problem. You may want to
adjust the MCA parameter pmix_server_max_wait and try again. If this
occurred during a connect/accept operation, you can adjust that time
using the pmix_base_exchange_timeout parameter.
--------------------------------------------------------------------------


Exception: MPI_ERR_UNKNOWN: unknown error

In [23]:
size, rank

(1, 0)